Code to detect screen from a video and get SLPM rate of flow using frames of the video as an input, accuracy cross verified using manually entered data

In [8]:
import cv2
import easyocr
import numpy as np
import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

# Image processing function to extract the region of interest (ROI) and recognize text
def process_image(image_path, output_folder, reader):
    image = cv2.imread(image_path)

    # preprocess the image
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 220, 255, cv2.THRESH_BINARY)

    # Find contours to detect the screen
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Initialize variables for the screen contour
    screen_contour = None

    # Loop through contours to find the screen based on aspect ratio and size
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        aspect_ratio = w / float(h)
        if 1.0 < aspect_ratio < 2.0 and 100 < w < 300:  # Adjust size and ratio as needed
            screen_contour = (x, y, w, h)
            break

    if screen_contour:
        x, y, w, h = screen_contour

        # Crop the region of interest (ROI)
        roi = image[y:y+h, x:x+w]

        # Optional: Further crop a specific part of the screen if needed
        specific_area = roi[48:48+45, 24:24+116]  # Adjust these values based on your needs

        # Save the cropped ROI image
        base_filename = os.path.splitext(os.path.basename(image_path))[0]
        roi_output_path = os.path.join(output_folder, f"{base_filename}_roi.jpg")
        cv2.imwrite(roi_output_path, roi)

        # Recognize text using EasyOCR
        results = reader.readtext(specific_area)

        recognized_text = ""
        confidence_scores = []
        for (bbox, text, confidence) in results:
            # Filter and process text to replace ',' with '.' and limit to 5 characters
            filtered_text = ''.join([char if char.isdigit() or char == '.' else '.' if char == ',' else '' for char in text])
            if filtered_text:
                recognized_text += filtered_text[:5]  # Limit to 5 characters
                confidence_scores.append(confidence)

        avg_confidence = np.mean(confidence_scores) if confidence_scores else 0
        return {"Image": os.path.basename(image_path), "Recognized Text": recognized_text.strip(), "Confidence": avg_confidence}
    else:
        return {"Image": os.path.basename(image_path), "Recognized Text": "Screen not detected", "Confidence": 0}

# Input folder containing images
folder_path = r"C:\Users\Pradhyumna R Shetty\Pictures\Internship 8th sem\Frames\Frames outlet flow"  # Replace with your input folder
output_folder = r"C:\Users\Pradhyumna R Shetty\Pictures\Internship 8th sem\sample\Output"  # Replace with your output folder
csv_output_path = r"C:\Users\Pradhyumna R Shetty\Pictures\Internship 8th sem\sample\final.csv"  # Replace with the CSV file path

# Ensure output folder exists
os.makedirs(output_folder, exist_ok=True)

# Initialize EasyOCR reader
reader = easyocr.Reader(['en'], gpu=True)

# Process each image in the folder and save results to the DataFrame
def process_images_in_folder():
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_image, os.path.join(folder_path, filename), output_folder, reader)
                   for filename in os.listdir(folder_path) if filename.lower().endswith(('.png', '.jpg', '.jpeg'))]
        results = [future.result() for future in futures]
    return results

# Process images and create a DataFrame
data = process_images_in_folder()
df = pd.DataFrame(data)

# Remove the '.jpg' extension from the 'Image' column
df['Image'] = df['Image'].str.replace('.jpg', '', regex=False)

# Filter out rows where the 'Recognized Text' column ends with '..'
df = df[~df['Recognized Text'].str.endswith('..', na=False)]

# Convert both columns to numeric values
df['Image'] = pd.to_numeric(df['Image'], errors='coerce')
df['Recognized Text'] = pd.to_numeric(df['Recognized Text'], errors='coerce')

# Sort the DataFrame by the 'Image' column
df_sorted = df.sort_values(by='Image')

# Filter out rows where 'Confidence' < 0.5
df_filtered = df_sorted

# Add a new column 'Time' at position 1 in time format (HH:MM:SS)
df_filtered.insert(1, 'Time', pd.to_timedelta(df_filtered['Image'] / 30, unit='s'))

# Apply the condition to modify 'Recognized Text'
df_filtered['Recognized Text'] = df_filtered['Recognized Text'].apply(
    lambda x: x / 10 if x > 1000 else x / 100 if x > 10000 else x
)

# Save the filtered and processed DataFrame to the CSV file
df_filtered.to_csv(csv_output_path, index=False, encoding='utf-8')

# Print the final DataFrame to verify
print(df_filtered)


import pandas as pd

# Example DataFrames
df1 = df_filtered
df2 = pd.read_csv(r"C:\Users\Pradhyumna R Shetty\Pictures\Internship 8th sem\True Values.csv")

# Accuracy calculation function with margin
def calculate_column_accuracy_with_margin(df1, df2, key_column, value_column_df1, value_column_df2, error_margin=0.001):

    # Inner join both DataFrames on the key column
    merged_df = pd.merge(df1, df2, on=key_column, how='inner')

    # Calculating the absolute difference between Recognized Text and Values and checking if it's within the margin
    merged_df['Within Margin'] = abs(merged_df[value_column_df1] - merged_df[value_column_df2]) <= error_margin * merged_df[value_column_df2]
    merged_df['Match'] = merged_df['Within Margin'].replace({True: 'Yes', False: 'No'})

    # Calculate accuracy
    matches = (merged_df['Match'] == 'Yes').sum()
    total = len(merged_df)
    accuracy = (matches / total) * 100 if total > 0 else 0

    return accuracy, merged_df

# Driver code to calculate accuracy and merge DataFrames
key_column = 'Image'
value_column_df1 = 'Recognized Text'
value_column_df2 = 'Values'
accuracy, merged_df = calculate_column_accuracy_with_margin(df1, df2, key_column, value_column_df1, value_column_df2)

# Print the accuracy
print(f"Accuracy: {accuracy:.2f}%")

# Print the merged DataFrame with the 'Match' column
print(merged_df)
merged_df.to_csv(r"C:\Users\Pradhyumna R Shetty\Pictures\Internship 8th sem\sample\2.csv", index=False, encoding='utf-8')


c:\Users\Pradhyumna R Shetty\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\modules\rnn.py:1123: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\cudnn\RNN.cpp:1410.)
  result = _VF.lstm(


      Image                      Time  Recognized Text  Confidence
0         0           0 days 00:00:00            463.2    0.475436
1         1 0 days 00:00:00.033333333            463.2    0.610356
1071      2 0 days 00:00:00.066666667            463.9    0.755070
1182      3    0 days 00:00:00.100000            463.3    0.433789
1404      5 0 days 00:00:00.166666667            463.0    0.366306
...     ...                       ...              ...         ...
1062   1954 0 days 00:01:05.133333333            463.9    0.260896
1063   1955 0 days 00:01:05.166666667            462.9    0.412159
1064   1956    0 days 00:01:05.200000            462.9    0.423156
1065   1957 0 days 00:01:05.233333333            462.9    0.423766
1066   1958 0 days 00:01:05.266666667            462.1    0.534634

[1831 rows x 4 columns]
Accuracy: 82.58%
      Image                      Time  Recognized Text  Confidence  Values  \
0         0           0 days 00:00:00            463.2    0.475436   463.2  

Same code but to take the video as an input

In [7]:
import cv2
import easyocr
import numpy as np
import os
import pandas as pd

# Image processing function to extract the region of interest (ROI) and recognize text
def process_frame(frame, reader, image_number):
    # Preprocess the image
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 220, 255, cv2.THRESH_BINARY)

    # Find contours to detect the screen
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Initialize variables for the screen contour
    screen_contour = None

    # Loop through contours to find the screen based on aspect ratio and size
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        aspect_ratio = w / float(h)
        if 1.0 < aspect_ratio < 2.0 and 100 < w < 300:  # Adjust size and ratio as needed
            screen_contour = (x, y, w, h)
            break

    if screen_contour:
        x, y, w, h = screen_contour

        # Crop the region of interest (ROI)
        roi = frame[y:y+h, x:x+w]

        # Optional: Further crop a specific part of the screen if needed
        specific_area = roi[48:48+45, 24:24+116]  # Adjust these values based on your needs

        # Recognize text using EasyOCR
        results = reader.readtext(specific_area)

        recognized_text = ""
        confidence_scores = []
        for (bbox, text, confidence) in results:
            # Filter and process text to replace ',' with '.' and limit to 5 characters
            filtered_text = ''.join([char if char.isdigit() or char == '.' else '.' if char == ',' else '' for char in text])
            if filtered_text:
                recognized_text += filtered_text[:5]  # Limit to 5 characters
                confidence_scores.append(confidence)

        avg_confidence = np.mean(confidence_scores) if confidence_scores else 0
        return {"Image": image_number, "Recognized Text": recognized_text.strip(), "Confidence": avg_confidence}
    else:
        return {"Image": image_number, "Recognized Text": "Screen not detected", "Confidence": 0}

# Video processing function to read frames from video
def process_video(video_path, reader):
    # Open the video file
    cap = cv2.VideoCapture(video_path)

    # Check if video is opened correctly
    if not cap.isOpened():
        print("Error: Could not open video file.")
        return []

    image_number = 0
    results = []

    # Process each frame of the video
    while True:
        ret, frame = cap.read()
        if not ret:
            break  # Stop if no frames are left

        image_number += 1
        result = process_frame(frame, reader, image_number)
        results.append(result)

    # Release the video capture object
    cap.release()
    return results

# Accuracy calculation function with margin
def calculate_column_accuracy_with_margin(df1, df2, key_column, value_column_df1, value_column_df2, error_margin=0.001):
    # Inner join both DataFrames on the key column
    merged_df = pd.merge(df1, df2, on=key_column, how='inner')

    # Calculating the absolute difference between Recognized Text and Values and checking if it's within the margin
    merged_df['Within Margin'] = abs(merged_df[value_column_df1] - merged_df[value_column_df2]) <= error_margin * merged_df[value_column_df2]
    merged_df['Match'] = merged_df['Within Margin'].replace({True: 'Yes', False: 'No'})

    # Calculate accuracy
    matches = (merged_df['Match'] == 'Yes').sum()
    total = len(merged_df)
    accuracy = (matches / total) * 100 if total > 0 else 0

    return accuracy, merged_df

# Main function
def main():
    # Input video file path
    video_path = r"C:\Users\Pradhyumna R Shetty\Pictures\Internship 8th sem\Videos\Outlet flow.mp4"  # Replace with your video file path
    csv_output_path = r"C:\Users\Pradhyumna R Shetty\Pictures\Internship 8th sem\sample\final.csv"  # Replace with the CSV file path

    # Initialize EasyOCR reader
    reader = easyocr.Reader(['en'], gpu=True)

    # Process the video and create a DataFrame
    data = process_video(video_path, reader)
    df = pd.DataFrame(data)

    # Filter out rows where the 'Recognized Text' column ends with '..'
    df = df[~df['Recognized Text'].str.endswith('..', na=False)]

    # Convert both columns to numeric values
    df['Image'] = pd.to_numeric(df['Image'], errors='coerce')
    df['Recognized Text'] = pd.to_numeric(df['Recognized Text'], errors='coerce')

    # Sort the DataFrame by the 'Image' column
    df_sorted = df.sort_values(by='Image')

    # Filter out rows where 'Confidence' < 0.5
    df_filtered = df_sorted

    # Add a new column 'Time' at position 1 in time format (HH:MM:SS)
    df_filtered.insert(1, 'Time', pd.to_timedelta(df_filtered['Image'] / 30, unit='s'))

    # Save the filtered and processed DataFrame to the CSV file
    df_filtered.to_csv(csv_output_path, index=False, encoding='utf-8')

    # Print the final DataFrame to verify
    print(df_filtered)

    # Read the second DataFrame (CSV file) for accuracy calculation
    df2 = pd.read_csv(r"C:\Users\Pradhyumna R Shetty\Pictures\Internship 8th sem\True Values.csv")

    # Accuracy calculation
    key_column = 'Image'  # Matching column: Image
    value_column_df1 = 'Recognized Text'
    value_column_df2 = 'Values'

    accuracy, merged_df = calculate_column_accuracy_with_margin(df_filtered, df2, key_column, value_column_df1, value_column_df2)

    # Print the accuracy
    print(f"Accuracy: {accuracy:.2f}%")

    # Print the merged DataFrame with the 'Match' column
    print(merged_df)

    # Save the merged results to a CSV
    merged_df.to_csv(r"C:\Users\Pradhyumna R Shetty\Pictures\Internship 8th sem\sample\merged_results.csv", index=False, encoding='utf-8')

# Run the main function
if __name__ == "__main__":
    main()


      Image                      Time  Recognized Text  Confidence
0         1 0 days 00:00:00.033333333            463.2    0.510558
1         2 0 days 00:00:00.066666667            463.2    0.617302
2         3    0 days 00:00:00.100000            463.9    0.750608
3         4 0 days 00:00:00.133333333            463.3    0.370143
5         6    0 days 00:00:00.200000            463.0    0.382352
...     ...                       ...              ...         ...
1954   1955 0 days 00:01:05.166666667            463.9    0.274256
1955   1956    0 days 00:01:05.200000            462.9    0.519958
1956   1957 0 days 00:01:05.233333333            462.9    0.849292
1957   1958 0 days 00:01:05.266666667            462.9    0.403921
1958   1959    0 days 00:01:05.300000            462.1    0.468410

[1829 rows x 4 columns]
Accuracy: 70.62%
      Image                      Time  Recognized Text  Confidence  Values  \
0         1 0 days 00:00:00.033333333            463.2    0.510558   463.2  

In [3]:
import cv2
import easyocr
import numpy as np
import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

# Image processing function to extract the region of interest (ROI) and recognize text
def process_image(image_path, output_folder, reader):
    image = cv2.imread(image_path)

    # Preprocess the image
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 210, 255, cv2.THRESH_BINARY)

    # Find contours to detect the screen
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Initialize variables for the screen contour
    screen_contour = None

    # Loop through contours to find the screen based on aspect ratio and size
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        aspect_ratio = w / float(h)
        if 1.0 < aspect_ratio < 2.0 and 100 < w < 300:  # Adjust size and ratio as needed
            screen_contour = (x, y, w, h)
            break

    if screen_contour:
        x, y, w, h = screen_contour

        # Crop the region of interest (ROI)
        roi = image[y:y+h, x:x+w]

        # Optional: Further crop a specific part of the screen if needed
        specific_area = roi[48:48+45, 24:24+116]  # Adjust these values based on your needs

        # Binarize the specific area
        gray_specific_area = cv2.cvtColor(specific_area, cv2.COLOR_BGR2GRAY)
        _, binarized_roi = cv2.threshold(gray_specific_area, 220, 255, cv2.THRESH_BINARY)

        # Save the binarized ROI image
        base_filename = os.path.splitext(os.path.basename(image_path))[0]
        binarized_output_path = os.path.join(output_folder, f"{base_filename}_binarized_roi.jpg")
        cv2.imwrite(binarized_output_path, binarized_roi)

        # Recognize text using EasyOCR
        results = reader.readtext(binarized_roi)

        recognized_text = ""
        confidence_scores = []
        for (bbox, text, confidence) in results:
            # Filter and process text to replace ',' with '.' and limit to 5 characters
            filtered_text = ''.join([char if char.isdigit() or char == '.' else '.' if char == ',' else '' for char in text])
            if filtered_text:
                recognized_text += filtered_text[:5]  # Limit to 5 characters
                confidence_scores.append(confidence)

        avg_confidence = np.mean(confidence_scores) if confidence_scores else 0
        return {"Image": os.path.basename(image_path), "Recognized Text": recognized_text.strip(), "Confidence": avg_confidence}
    else:
        return {"Image": os.path.basename(image_path), "Recognized Text": "Screen not detected", "Confidence": 0}

# Input folder containing images
folder_path = r"C:\Users\Pradhyumna R Shetty\Pictures\Internship 8th sem\Frames\Frames outlet flow"  # Replace with your input folder
output_folder = r"C:\Users\Pradhyumna R Shetty\Pictures\Internship 8th sem\sample\Output"  # Replace with your output folder
csv_output_path = r"C:\Users\Pradhyumna R Shetty\Pictures\Internship 8th sem\sample\final.csv"  # Replace with the CSV file path

# Ensure output folder exists
os.makedirs(output_folder, exist_ok=True)

# Initialize EasyOCR reader
reader = easyocr.Reader(['en'], gpu=True)

# Process each image in the folder and save results to the DataFrame
def process_images_in_folder():
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_image, os.path.join(folder_path, filename), output_folder, reader)
                   for filename in os.listdir(folder_path) if filename.lower().endswith(('.png', '.jpg', '.jpeg'))]
        results = [future.result() for future in futures]
    return results

# Process images and create a DataFrame
data = process_images_in_folder()
df = pd.DataFrame(data)

# Remove the '.jpg' extension from the 'Image' column
df['Image'] = df['Image'].str.replace('.jpg', '', regex=False)

# Filter out rows where the 'Recognized Text' column ends with '..'
df = df[~df['Recognized Text'].str.endswith('..', na=False)]

# Convert both columns to numeric values
df['Image'] = pd.to_numeric(df['Image'], errors='coerce')
df['Recognized Text'] = pd.to_numeric(df['Recognized Text'], errors='coerce')

# Sort the DataFrame by the 'Image' column
df_sorted = df.sort_values(by='Image')

# Filter out rows where 'Confidence' < 0.5
df_filtered = df_sorted[df_sorted['Confidence'] >= 0.5]

# Add a new column 'Time' at position 1 in time format (HH:MM:SS)
df_filtered.insert(1, 'Time', pd.to_timedelta(df_filtered['Image'] / 30, unit='s'))

# Apply the condition to modify 'Recognized Text'
df_filtered['Recognized Text'] = df_filtered['Recognized Text'].apply(
    lambda x: x / 10 if x > 1000 else x / 100 if x > 10000 else x
)

# Save the filtered and processed DataFrame to the CSV file
df_filtered.to_csv(csv_output_path, index=False, encoding='utf-8')

# Print the final DataFrame to verify
print(df_filtered)


c:\Users\Pradhyumna R Shetty\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\modules\rnn.py:1123: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\cudnn\RNN.cpp:1410.)
  result = _VF.lstm(


      Image                      Time  Recognized Text  Confidence
0         0           0 days 00:00:00           4632.1    0.513725
1         1 0 days 00:00:00.033333333            463.3    0.532615
1182      3    0 days 00:00:00.100000           4639.8    0.636152
1293      4 0 days 00:00:00.133333333            463.8    0.933193
446      14 0 days 00:00:00.466666667            464.0    0.994625
...     ...                       ...              ...         ...
1044   1938    0 days 00:01:04.600000            468.9    0.609952
1045   1939 0 days 00:01:04.633333333           4688.8    0.621501
1055   1948 0 days 00:01:04.933333333            465.9    0.576294
1064   1956    0 days 00:01:05.200000            462.9    0.867372
1065   1957 0 days 00:01:05.233333333            462.9    0.885753

[527 rows x 4 columns]


C:\Users\Pradhyumna R Shetty\AppData\Local\Temp\ipykernel_41340\3438992163.py:108: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['Recognized Text'] = df_filtered['Recognized Text'].apply(
